# Gold crowns: sample HOT trees and cut crops for hand measurement

Run on a CPU runtime. Set `REPO_SHA` in the next cell, then run the cells in order.

The sample is stratified by HOT split and by the mapper's `diameter_crown` (see `experiments/gold_crowns.py`). Every tree gets a 20 m crop centred on its point. The output on Drive has two parts:

- `annotate.zip`: the crops, `items.json`, `items_second.json` and `gold_annotator.html`. It holds no tags, so it can go to annotators as it is. Unzip it, open `gold_annotator.html` in Chrome or Edge, and pick the unzipped folder.
- `key/manifest.csv`: the tags, strata and splits per `gold_id`. Keep it away from annotators until the drawing is done.

The repo is public; this notebook still clones it with a read-only GitHub token from Colab Secrets (the key icon), named `GITHUB_TOKEN`, with notebook access on.


In [ ]:
REPO_SHA = ""
SEED = 0  # fixes the sample, the item order and the second-annotator subset
OUT_NAME = "gold_crowns"  # folder under the Drive root; must not exist yet

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

assert REPO_SHA, "set REPO_SHA to the pushed commit"
# a read-only GitHub token in Colab Secrets (key icon), named GITHUB_TOKEN
REPO = "github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/fair_trees
# git clone through subprocess so the token never reaches the cell output
url = f"https://x-access-token:{token}@{REPO}"
clone = subprocess.run(["git", "clone", "-q", url, "/content/fair_trees"])
# checked by hand: a CalledProcessError message would print the URL, token included
if clone.returncode:
    raise RuntimeError(f"git clone failed (exit {clone.returncode}): check the GITHUB_TOKEN secret")
# drop the token from the saved remote so it is not left in .git/config
git = ["git", "-C", "/content/fair_trees"]
subprocess.run([*git, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
subprocess.run([*git, "checkout", "-q", REPO_SHA], check=True)
del token, url

%pip install /content/fair_trees -r /content/fair_trees/experiments/requirements.txt
sys.path.insert(0, "/content/fair_trees")

In [ ]:
from huggingface_hub import hf_hub_download

from experiments.label_study.hot import REPO_ID, REVISION, read_hot

FILES = {
    "labels": "stac/project-50661/labels.parquet",
    "tiles": "stac/project-50661/tiles.parquet",
}
# crops reach into neighbouring tiles of any split, so all three splits are read
IMAGES = [
    "data/train-00000-of-00002.parquet",
    "data/train-00001-of-00002.parquet",
    "data/validation-00000-of-00001.parquet",
    "data/test-00000-of-00001.parquet",
]


def fetch(path):
    return Path(
        hf_hub_download(REPO_ID, path, repo_type="dataset", revision=REVISION, local_dir="/content/hot_raw")
    )


data = read_hot(fetch(FILES["labels"]), fetch(FILES["tiles"]))
image_files = [fetch(p) for p in IMAGES]
print(len(data.trees), "trees,", len(data.groups), "groups,", len(data.tiles), "tiles")

In [ ]:
import json
import shutil

import pyarrow.parquet as pq

from experiments.gold_crowns import sample_gold, write_gold


def images():
    """Yield (tile_id, image bytes) for every HOT tile, one record batch at a time."""
    for path in image_files:
        for batch in pq.ParquetFile(path).iter_batches(columns=["tile_id", "image"], batch_size=64):
            for row in batch.to_pylist():
                yield row["tile_id"], row["image"]["bytes"]


sample = sample_gold(data, seed=SEED)
local = Path("/content") / OUT_NAME
shutil.rmtree(local, ignore_errors=True)  # scratch on the Colab VM only
stats = write_gold(sample, data, images(), local, seed=SEED)
stats["repo_sha"], stats["seed"] = REPO_SHA, SEED
(local / "key" / "stats.json").write_text(json.dumps(stats, indent=2))
print(json.dumps(stats, indent=2))

In [ ]:
from IPython.display import Image, display
from PIL import Image as PILImage

# a contact sheet of the first 24 crops, as the annotator will see them (no tags)
items = json.loads((local / "annotate" / "items.json").read_text())["items"][:24]
THUMB = 200
contact = PILImage.new("RGB", (6 * THUMB, 4 * THUMB))
for k, item in enumerate(items):
    crop = PILImage.open(local / "annotate" / item["file"]).resize((THUMB, THUMB))
    contact.paste(crop, ((k % 6) * THUMB, (k // 6) * THUMB))
contact.save(local / "key" / "contact.png")
display(Image(filename=str(local / "key" / "contact.png")))

In [ ]:
out = DRIVE / OUT_NAME
assert not out.exists(), f"{out} exists: keep the frozen set, pick another OUT_NAME"
out.mkdir(parents=True)
shutil.make_archive(str(out / "annotate"), "zip", local / "annotate")
shutil.copytree(local / "key", out / "key")
print(sorted(p.name for p in out.rglob("*") if p.is_file()))
drive.flush_and_unmount()